# Classifying

In [86]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, matthews_corrcoef, f1_score, accuracy_score
from mlxtend.classifier import OneRClassifier
from sklearn.neighbors import KNeighborsClassifier
import plotly.express as px
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


## Datasets

In [87]:
def train_test_val(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X.to_numpy(), y.to_numpy(), test_size=0.4, random_state=42)
    X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
    return X_train, X_test, X_val, y_train, y_test, y_val


### Numerical

In [88]:
df_numerical = pd.read_pickle("../data/skin-lesion-data/numerical.pickle")

In [89]:
Xn = df_numerical.drop("malignant", axis=1)
yn = df_numerical["malignant"]

In [90]:
Xn_train, Xn_test, Xn_val, yn_train, yn_test, yn_val = train_test_val(Xn, yn)

In [91]:
Xn_train.shape, yn_train.shape

((463, 56), (463,))

In [92]:
Xn_val.shape, yn_val.shape

((155, 56), (155,))

In [93]:
Xn_test.shape, yn_test.shape

((155, 56), (155,))

### Categorical

In [94]:
df_categorical = pd.read_pickle("../data/skin-lesion-data/categorical.pickle")

In [95]:
Xc = df_categorical.drop("malignant", axis=1)
yc = df_categorical["malignant"]

In [96]:
Xc_train, Xc_test, Xc_val, yc_train, yc_test, yc_val = train_test_val(Xc, yc)

In [97]:
Xc_train.shape, yc_train.shape

((463, 22), (463,))

In [98]:
Xc_val.shape, yc_val.shape

((155, 22), (155,))

In [99]:
Xc_test.shape, yc_test.shape

((155, 22), (155,))

### Ordinal

In [100]:
df_ordinal = pd.read_pickle("../data/skin-lesion-data/ordinal.pickle")

In [101]:
Xo = df_ordinal.drop("malignant", axis=1)
yo = df_ordinal["malignant"]

In [102]:
Xo_train, Xo_test, Xo_val, yo_train, yo_test, yo_val = train_test_val(Xo, yo)

In [103]:
Xo_train.shape, yo_train.shape

((463, 22), (463,))

In [104]:
Xo_val.shape, yo_val.shape

((155, 22), (155,))

In [105]:
Xo_test.shape, yo_test.shape

((155, 22), (155,))

## Accuracy metrics

> Note that in binary classification, recall of the positive class is also known as “sensitivity”; recall of the negative class is “specificity”.

> [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

In [106]:
def train_test(model, X_train, y_train, X_test, y_test):
    # Train
    model.fit(X_train, y_train)
    y_pred = model.predict(X_train)

    # Test
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    print("MCC", matthews_corrcoef(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    fig = px.imshow(cm, text_auto=True)
    fig.show()


In [107]:
def train_val(models, X_train, y_train, X_val, y_val):
    accuracies = []
    f1s = []
    sensitivities = []
    specificities = []
    mccs = []

    for model in models:
        # Train
        model.fit(X_train, y_train)

        # Val
        y_pred = model.predict(X_val)

        accuracy = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)
        mcc = matthews_corrcoef(y_val, y_pred)

        accuracies.append(accuracy)
        f1s.append(f1)
        sensitivities.append(sensitivity)
        specificities.append(specificity)
        mccs.append(mcc)

    return accuracies, f1s, sensitivities, specificities, mccs

In [108]:
def val_plot(label, xs, accuracies, f1s, sensitivities, specificities, mccs):
    df = pd.DataFrame({
        label: xs,
        "Accuracy": accuracies,
        "F1": f1s,
        "Sensitivity": sensitivities,
        "Specificity": specificities,
        "MCC": mccs,
    })

    df_melted = df.melt(id_vars=label, value_vars=["Accuracy", "F1", "Sensitivity", "Specificity", "MCC"],
                        var_name="Metric", value_name="Score")

    # Plot
    fig = px.line(
        df_melted,
        x=label,
        y="Score",
        color="Metric",
        markers=True,
        title=f"Val scores vs {label}"
    )
    fig.show()

## ZeroR


In [109]:
zeror = DummyClassifier(strategy="most_frequent")
train_test(zeror, Xc_train, yc_train, Xc_test, yc_test)

              precision    recall  f1-score   support

       False       0.00      0.00      0.00        50
        True       0.68      1.00      0.81       105

    accuracy                           0.68       155
   macro avg       0.34      0.50      0.40       155
weighted avg       0.46      0.68      0.55       155

MCC 0.0


/Users/oriol/Documents/heriot-watt/f21dl/group-coursework-meddm/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/oriol/Documents/heriot-watt/f21dl/group-coursework-meddm/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/oriol/Documents/heriot-watt/f21dl/group-coursework-meddm/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



## OneR

In [110]:
oner = OneRClassifier()
train_test(oner, Xc_train, yc_train, Xc_test, yc_test)

              precision    recall  f1-score   support

       False       1.00      0.36      0.53        50
        True       0.77      1.00      0.87       105

    accuracy                           0.79       155
   macro avg       0.88      0.68      0.70       155
weighted avg       0.84      0.79      0.76       155

MCC 0.5252736513086528


## KNN

In [111]:
ks = list(range(1, 50))
models = [KNeighborsClassifier(n_neighbors=k) for k in ks]
accuracies, f1s, sensitivities, specificities, mccs = train_val(models, Xn_train, yn_train, Xn_val, yn_val)
val_plot("K", ks, accuracies, f1s, sensitivities, specificities, mccs)

In [112]:
K = 13
knn = KNeighborsClassifier(n_neighbors=K)
train_test(knn, Xn_train, yn_train, Xn_test, yn_test)

              precision    recall  f1-score   support

           0       0.74      0.58      0.65        50
           1       0.82      0.90      0.86       105

    accuracy                           0.80       155
   macro avg       0.78      0.74      0.76       155
weighted avg       0.79      0.80      0.79       155

MCC 0.5222119873904899


## Gaussian Naïve Bayes

In [113]:
nb = GaussianNB()
train_test(nb, Xn_train, yn_train, Xn_test, yn_test)

              precision    recall  f1-score   support

           0       0.82      0.46      0.59        50
           1       0.79      0.95      0.86       105

    accuracy                           0.79       155
   macro avg       0.80      0.71      0.73       155
weighted avg       0.80      0.79      0.77       155

MCC 0.5010688131796587


## Decision Tree

In [116]:
depths = list(range(1, 50))
models = [DecisionTreeClassifier(max_depth=d, random_state=42) for d in depths]
accuracies, f1s, sensitivities, specificities, mccs = train_val(models, Xo_train, yo_train, Xo_val, yo_val)
val_plot("Depth", depths, accuracies, f1s, sensitivities, specificities, mccs)

In [115]:
max_depth = 7
dt = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
train_test(dt, Xn_train, yn_train, Xn_test, yn_test)

              precision    recall  f1-score   support

           0       0.76      0.64      0.70        50
           1       0.84      0.90      0.87       105

    accuracy                           0.82       155
   macro avg       0.80      0.77      0.78       155
weighted avg       0.82      0.82      0.81       155

MCC 0.5729576395558952


## LogisticRegression

In [117]:
iters = list(range(1, 100))
models = [LogisticRegression(max_iter=i, random_state=42) for i in iters]
accuracies, f1s, sensitivities, specificities, mccs = train_val(models, Xn_train, yn_train, Xn_val, yn_val)
val_plot("Iterations", iters, accuracies, f1s, sensitivities, specificities, mccs)

/Users/oriol/Documents/heriot-watt/f21dl/group-coursework-meddm/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning:

lbfgs failed to converge after 1 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

/Users/oriol/Documents/heriot-watt/f21dl/group-coursework-meddm/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning:

lbfgs failed to converge after 2 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2).
You might also want to scale the data as shown in:
    htt

In [118]:
max_iter = 20
logreg = LogisticRegression(max_iter=max_iter, random_state=42)
train_test(logreg, Xn_train, yn_train, Xn_test, yn_test)

              precision    recall  f1-score   support

           0       0.80      0.70      0.74        50
           1       0.86      0.91      0.89       105

    accuracy                           0.85       155
   macro avg       0.83      0.81      0.82       155
weighted avg       0.84      0.85      0.84       155

MCC 0.6368867879182144


/Users/oriol/Documents/heriot-watt/f21dl/group-coursework-meddm/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning:

lbfgs failed to converge after 20 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=20).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

